# Pupil Size Statistical Evaluation and Metrics Definition
# 瞳孔径統計評価と評価指標定義

This notebook processes the cleaned pupil size data generated by the EyeLink ASC parsing and interpolation pipeline, calculates the key statistical evaluation metrics (Baseline, Normalized Constriction, Early AUC, Late AUC, and 6s PIPR), and visualizes the results.

このノートブックは、EyeLinkの `.asc` ファイルからパース・補間されたクリーンな瞳孔径データを読み込み、主要な評価指標（ベースライン、正規化収縮率、Early AUC、Late AUC、6s PIPR）を計算・視覚化します。

---

## Mathematical Formulation & Definitions / 数学的定式化と定義

### 1. Baseline Pupil Diameter ($A_{base}$) / ベースライン瞳孔径
The median pupil diameter of the cleaned pupil size during the 1.5 seconds immediately preceding the onset of the light stimulus (initial illumination). Using the **median** rather than the mean is highly advantageous to eliminate the influence of blink-related remnants or transient noise.

刺激光提示（初期照射）の直前1.5秒間における、補間された瞳孔径の**中央値**。瞬き直後のノイズなどの外れ値（極端な値）の影響を排除するために、平均値ではなく中央値を採用しています。
$$A_{base} = \text{median}(\{ A_t \mid T_{onset} - 1.5 \le t < T_{onset} \})$$

### 2. Normalized Constriction at time $t$ / 正規化瞳孔収縮率
Because of individual differences in baseline pupil size, raw pupil sizes are converted into a dimensionless constriction ratio:

瞳孔の基準サイズには個人差があるため、基準に対する比率（無次元比）に変換して評価します。
$$NormalizedConstriction_t = \frac{A_{base} - A_t}{A_{base}}$$

### 3. Early AUC (0–6 seconds) / アーリーAUC（0〜6秒）
The integral value (area under the curve) from the complete offset of the stimulus block ($T_{offset}$) up to 6 seconds. This time window is a transitional phase where the fast cone/rod response decay and the slow ipRGC activation overlap.

刺激ブロックの完全な消灯（$T_{offset}$）から6秒後までの積分値（曲線下面積）。この区間は、錐体・桿体細胞の減衰反応と、ipRGC（メラノプシン）の活性化開始が混在する過渡期です。
$$EarlyAUC = \int_{0}^{6} \frac{A_{base} - A_t}{A_{base}} dt$$

### 4. Late AUC (6–12 seconds) / レイトAUC（6〜12秒）
The integral value from 6 to 12 seconds post-stimulus offset. This is the most critical phase where the cone/rod responses have almost completely decayed, purely reflecting the sustained response driven by ipRGC melanopsin bistability.

消灯後6秒から12秒までの積分値。錐体・桿体細胞の応答がほぼ完全に減衰し、ipRGC（メラノプシン双安定性機能）の持続的な瞳孔収縮反応のみが純粋に現れる最も重要なフェーズです。
$$LateAUC = \int_{6}^{12} \frac{A_{base} - A_t}{A_{base}} dt$$

### 5. 6s PIPR (Post-Illumination Pupil Response) / 6秒PIPR
The normalized constriction at exactly 6 seconds after the light turns off.

消灯からちょうど6秒後の時点における正規化瞳孔収縮率。
$$PIPR_{6s} = NormalizedConstriction_{T_{offset} + 6.0}$$


In [ ]:
# Install japanize-matplotlib for Japanese labels support in Google Colab
# Google Colabで日本語フォントを使用するためにインストールします
!pip install -q japanize-matplotlib

In [ ]:
import os
import glob
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import japanize_matplotlib
except ImportError:
    pass

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 7)
plt.rcParams["font.family"] = "IPAexGothic"  # Fallback for standard Japanese environment

In [ ]:
# Environment Detection (Google Colab vs Local Mode)
# 実行環境（Google Colabまたはローカル環境）の自動検出
try:
    from google.colab import files
    IN_COLAB = True
    print("Running on Google Colab. Ready to upload files!")
    print("Google Colabで実行中。ファイルをアップロードしてください。")
except ImportError:
    IN_COLAB = False
    print("Running locally. Local mode enabled!")
    print("ローカル環境で実行中。ローカルモードが有効になりました。")

In [ ]:
def load_cleaned_data(file_source):
    """
    Loads pupil data from a file path or in-memory BytesIO buffer,
    supporting both CSV and Excel formats.
    """
    if isinstance(file_source, bytes):
        # Uploaded bytes in Colab
        try:
            df = pd.read_csv(io.BytesIO(file_source))
        except Exception:
            df = pd.read_excel(io.BytesIO(file_source))
    else:
        # Local filepath
        ext = os.path.splitext(file_source)[1].lower()
        if ext == '.csv':
            df = pd.read_csv(file_source)
        elif ext in ['.xls', '.xlsx', '.xlsm']:
            df = pd.read_excel(file_source)
        else:
            raise ValueError(f"Unsupported file type: {ext}")
    return df

In [ ]:
def integrate_auc(y, x):
    """
    Calculates Area Under the Curve (AUC) using trapezoidal integration.
    This is a robust manual implementation to avoid deprecation warnings or version issues with trapz/trapezoid.
    """
    if len(x) < 2:
        return 0.0
    x = np.array(x)
    y = np.array(y)
    dx = np.diff(x)
    y_mean = (y[:-1] + y[1:]) / 2.0
    return np.sum(y_mean * dx)

def calculate_pupil_metrics(df, filename_label=""):
    """
    Calculates experimental metrics:
    - Stimulus onset (T_onset) and offset (T_offset)
    - Baseline Pupil Size (A_base) (median of 1.5s window before T_onset)
    - Normalized Constriction at all time points
    - Early AUC (0-6s post-stimulus offset)
    - Late AUC (6-12s post-stimulus offset)
    - 6s PIPR (at exactly T_offset + 6.0s)
    """
    # 1. Stimulus Onset and Offset Detection from 'blue_active' transitions
    diff = df['blue_active'].diff().fillna(0)
    onset_indices = df[diff == 1].index
    offset_indices = df[diff == -1].index
    
    if len(onset_indices) > 0:
        T_onset = df.loc[onset_indices[0], 'elasped_sec']
    else:
        T_onset = df[df['blue_active'] == 1]['elasped_sec'].min()
        if pd.isna(T_onset):
            raise ValueError(f"Could not find stimulus onset in {filename_label} (blue_active is always 0).")
            
    if len(offset_indices) > 0:
        T_offset = df.loc[offset_indices[-1], 'elasped_sec']
    else:
        T_offset = df[df['blue_active'] == 1]['elasped_sec'].max()
        if pd.isna(T_offset):
            T_offset = df['elasped_sec'].max()
            
    # 2. Baseline Pupil Size (A_base) - Median of the 1.5 seconds immediately preceding T_onset
    baseline_mask = (df['elasped_sec'] >= T_onset - 1.5) & (df['elasped_sec'] < T_onset)
    baseline_df = df[baseline_mask]
    
    if len(baseline_df) == 0:
        # Fallback if recording starts late
        baseline_df = df[df['elasped_sec'] < T_onset]
        
    A_base = baseline_df['cleaned_pupil_size'].dropna().median()
    if pd.isna(A_base):
        # Fallback if no baseline samples exist
        A_base = df['cleaned_pupil_size'].dropna().median()
        
    # 3. Normalized Constriction time series
    df_out = df.copy()
    df_out['normalized_constriction'] = (A_base - df_out['cleaned_pupil_size']) / A_base
    
    # 4. Early AUC (0-6s post-stimulus offset)
    early_mask = (df_out['elasped_sec'] >= T_offset) & (df_out['elasped_sec'] <= T_offset + 6.0)
    early_df = df_out[early_mask]
    early_auc = integrate_auc(early_df['normalized_constriction'], early_df['elasped_sec'])
    
    # 5. Late AUC (6-12s post-stimulus offset)
    late_mask = (df_out['elasped_sec'] >= T_offset + 6.0) & (df_out['elasped_sec'] <= T_offset + 12.0)
    late_df = df_out[late_mask]
    late_auc = integrate_auc(late_df['normalized_constriction'], late_df['elasped_sec'])
    
    # 6. 6s PIPR (at exactly T_offset + 6.0s)
    target_time = T_offset + 6.0
    idx_closest = (df_out['elasped_sec'] - target_time).abs().idxmin()
    pipr_6s = df_out.loc[idx_closest, 'normalized_constriction']
    closest_time = df_out.loc[idx_closest, 'elasped_sec']
    
    # Safeguard if recording ends before 6s post-stimulus
    if abs(closest_time - target_time) > 0.1:
        pipr_6s = np.nan
        
    metrics = {
        "filename": os.path.basename(filename_label),
        "T_onset_sec": T_onset,
        "T_offset_sec": T_offset,
        "Baseline_Abase": A_base,
        "Early_AUC_0_6": early_auc,
        "Late_AUC_6_12": late_auc,
        "PIPR_6s": pipr_6s,
        "PIPR_time_sec": closest_time
    }
    return metrics, df_out

In [ ]:
def plot_pupil_metrics(df_metrics, df_processed, title_prefix=""):
    """
    Plots raw vs cleaned pupil size (top) and normalized constriction (bottom)
    with baseline, stimulus, Early AUC, and Late AUC phases clearly color-shaded.
    Also overlays the normalized constriction line and AUC windows on the top (unflipped) graph.
    """
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 12), sharex=True)
    
    t = df_processed['elasped_sec']
    raw_pupil = df_processed['pupil_size']
    clean_pupil = df_processed['cleaned_pupil_size']
    norm_const = df_processed['normalized_constriction']
    
    T_onset = df_metrics['T_onset_sec']
    T_offset = df_metrics['T_offset_sec']
    A_base = df_metrics['Baseline_Abase']
    
    # --- Panel 1: Pupil Size over Time (Unflipped) ---
    # Left Y-axis: Pupil Size
    ax1.plot(t, raw_pupil, label='生データの瞳孔径 (Raw Pupil Size)', color='lightgray', alpha=0.8, linewidth=1)
    ax1.plot(t, clean_pupil, label='補間後の瞳孔径 (Cleaned Pupil Size)', color='darkgreen', linewidth=1.5)
    ax1.axhline(A_base, color='red', linestyle='--', linewidth=1.5, label=f'ベースライン A_base ({A_base:.1f})')
    
    ax1.set_ylabel("瞳孔径 (Pupil Size, raw units)", color="darkgreen", fontsize=12)
    ax1.tick_params(axis='y', labelcolor="darkgreen")
    
    # Right Y-axis: Normalized Constriction
    ax1_twin = ax1.twinx()
    ax1_twin.plot(t, norm_const, label='収縮率 (Normalized Constriction)', color='darkorange', linewidth=1.2, linestyle=':')
    ax1_twin.set_ylabel("正規化収縮率 (Normalized Constriction)", color="darkorange", fontsize=12)
    ax1_twin.tick_params(axis='y', labelcolor="darkorange")
    ax1_twin.grid(False) # avoid overlapping grid lines
    
    # Shade windows on Panel 1
    # Baseline
    ax1.axvspan(T_onset - 1.5, T_onset, color='gray', alpha=0.15, label='ベースライン算出窓 (1.5s Baseline Window)')
    # Stimulus block
    ax1.axvspan(T_onset, T_offset, color='blue', alpha=0.08, label='青色光提示フェーズ (Blue LED Active)')
    # Early AUC (0-6s)
    ax1.axvspan(T_offset, T_offset + 6.0, color='green', alpha=0.08, label='Early AUC Window (0-6s)')
    # Late AUC (6-12s)
    ax1.axvspan(T_offset + 6.0, T_offset + 12.0, color='purple', alpha=0.08, label='Late AUC Window (6-12s)')
    
    # Mark 6s PIPR on Panel 1 twin axis
    pipr_t = df_metrics['PIPR_time_sec']
    pipr_y = df_metrics['PIPR_6s']
    if not pd.isna(pipr_y):
        ax1_twin.scatter(pipr_t, pipr_y, color='red', s=80, zorder=5, label=f'6s PIPR ({pipr_y:.4f})')
        ax1.axvline(T_offset + 6.0, color='red', linestyle=':', linewidth=1.5)
        
    ax1.set_title(f"{title_prefix} - 瞳孔径と正規化収縮率の経時変化 (Pupil Size & Normalized Constriction)", fontsize=14, fontweight='bold')
    
    # Combine legends for ax1 and ax1_twin
    lines, labels = ax1.get_legend_handles_labels()
    lines_twin, labels_twin = ax1_twin.get_legend_handles_labels()
    ax1.legend(lines + lines_twin, labels + labels_twin, loc='lower right')
    
    # --- Panel 2: Normalized Constriction Timeline ---
    ax2.plot(t, norm_const, label='収縮率 (Normalized Constriction)', color='darkorange', linewidth=1.5)
    
    # Shade windows on Panel 2
    ax2.axvspan(T_offset, T_offset + 6.0, color='green', alpha=0.1, label='Early AUC Window (0-6s)')
    ax2.axvspan(T_offset + 6.0, T_offset + 12.0, color='purple', alpha=0.1, label='Late AUC Window (6-12s)')
    
    # Mark 6s PIPR on Panel 2
    if not pd.isna(pipr_y):
        ax2.scatter(pipr_t, pipr_y, color='red', s=120, zorder=5, label=f'6s PIPR ({pipr_y:.4f})')
        ax2.axvline(T_offset + 6.0, color='red', linestyle=':', linewidth=1.5)
        
    # Text box for calculated metrics
    metrics_txt = (
        f"Baseline A_base: {A_base:.1f}\n"
        f"Early AUC (0-6s): {df_metrics['Early_AUC_0_6']:.4f}\n"
        f"Late AUC (6-12s): {df_metrics['Late_AUC_6_12']:.4f}\n"
        f"6s PIPR: {pipr_y:.4f}"
    )
    ax2.text(0.02, 0.95, metrics_txt, transform=ax2.transAxes, fontsize=11, fontweight='bold',
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
             
    ax2.set_title(f"{title_prefix} - 正規化収縮率と評価指標 (Normalized Constriction & Metrics)", fontsize=14, fontweight='bold')
    ax2.set_xlabel("経過時間 (Elapsed Time, seconds)", fontsize=12)
    ax2.set_ylabel("収縮率 (Normalized Constriction)", fontsize=12)
    ax2.legend(loc='lower right')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Step 1: Load, Process, and Visualize a single target file
# ステップ 1: 単一対象ファイルの読み込み、処理、および可視化

if IN_COLAB:
    print("Please upload a cleaned pupil CSV or Excel file:")
    uploaded_files = files.upload()
    uploaded_names = list(uploaded_files.keys())
    
    if len(uploaded_names) > 0:
        target_file = uploaded_names[0]
        df_single = load_cleaned_data(uploaded_files[target_file])
    else:
        df_single = None
        target_file = None
        print("No file uploaded.")
else:
    # Local mode - Prompts user for file path matching Colab parser behavior
    target_file = input('Cleaned CSV/Excel filepath: ')
    
    print(f"Running locally. Targeting file: {target_file}")
    print(f"ローカル環境で実行中。対象ファイル: {target_file}")
    
    if not os.path.exists(target_file):
        raise FileNotFoundError(
            f"Could not find cleaned file at path: '{target_file}'. "
            f"Please ensure the file exists."
        )
    df_single = load_cleaned_data(target_file)

if df_single is not None:
    metrics_single, df_proc_single = calculate_pupil_metrics(df_single, target_file)
    print(f"\nCalculated Metrics for {os.path.basename(target_file)}:")
    for k, v in metrics_single.items():
        print(f"  {k}: {v}")
    plot_pupil_metrics(metrics_single, df_proc_single, os.path.basename(target_file))


In [ ]:
# Step 2: Export processed metrics for the single dataset
# ステップ 2: 処理された単一データセットの指標エクスポート

if 'df_single' in globals() and df_single is not None:
    summary_df = pd.DataFrame([metrics_single])
    
    # Format and present summary table
    display_df = summary_df[[
        "filename", "Baseline_Abase", "Early_AUC_0_6", "Late_AUC_6_12", "PIPR_6s"
    ]].rename(columns={
        "filename": "ファイル名 (Filename)",
        "Baseline_Abase": "ベースライン (A_base)",
        "Early_AUC_0_6": "Early AUC (0-6秒)",
        "Late_AUC_6_12": "Late AUC (6-12秒)",
        "PIPR_6s": "6秒 PIPR"
    })
    
    print("\n=== SUMMARY RESULT / 処理結果 ===")
    from IPython.display import display, HTML
    display(HTML(display_df.to_html(index=False)))
    
    # Local save paths
    out_dir = "./out"
    if not os.path.exists(out_dir):
        out_dir = "out"
    os.makedirs(out_dir, exist_ok=True)
    
    base_name = os.path.splitext(os.path.basename(target_file))[0].replace('_cleaned', '')
    csv_out = os.path.join(out_dir, f"{base_name}_metrics.csv")
    xlsx_out = os.path.join(out_dir, f"{base_name}_metrics.xlsx")
    
    summary_df.to_csv(csv_out, index=False)
    summary_df.to_excel(xlsx_out, index=False)
    print(f"\nSaved summary results to:\n  - {csv_out}\n  - {xlsx_out}")
    
    if IN_COLAB:
        print("\nDownloading summary reports directly to your computer...")
        files.download(csv_out)
        files.download(xlsx_out)
else:
    print("No data available to export. Please run Step 1 first.")
